# TCGA-BRCA Baseline Model-Input V1 Review

This notebook reviews the saved baseline model-input v1 outputs from disk only. It does not parse raw files, freeze the endpoint, add treatment detail, or perform model training.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


def detect_repo_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Unable to locate the repository root from the notebook path.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = detect_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'model-input'
    / 'tcga_brca_baseline_model_input_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest baseline model-input pointer not found: {latest_pointer_path}. Run the model-input script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
matrix_path = repo_root / latest_pointer['baseline_model_input_v1_tsv']
feature_dictionary_path = repo_root / latest_pointer['baseline_model_input_v1_feature_dictionary_tsv']
encoding_spec_path = repo_root / latest_pointer['baseline_model_input_v1_encoding_spec_tsv']
missingness_actions_path = repo_root / latest_pointer['baseline_model_input_v1_missingness_actions_tsv']
summary_path = repo_root / latest_pointer['baseline_model_input_v1_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']
results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)

matrix_df = read_tsv(matrix_path)
feature_dictionary_df = read_tsv(feature_dictionary_path)
encoding_spec_df = read_tsv(encoding_spec_path)
missingness_actions_df = read_tsv(missingness_actions_path)
summary_df = read_tsv(summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

if not bool(run_log.get('validation', {}).get('passed', False)):
    raise ValueError('run_log.json does not report validation.passed == true.')
if matrix_df.empty:
    raise ValueError('baseline_model_input_v1.tsv contains no rows.')
if feature_dictionary_df['model_input_field_name'].nunique() != matrix_df.shape[1]:
    raise ValueError('Feature dictionary output-column count did not match the saved matrix column count.')
if set(encoding_spec_df['output_field_name']) != set(matrix_df.columns):
    raise ValueError('Encoding spec output fields did not reconcile to the saved matrix columns.')


In [ ]:
matrix_shape_df = pd.DataFrame(
    [
        {
            'baseline_model_input_v1_run_id': latest_pointer['baseline_model_input_v1_run_id'],
            'matrix_row_count': matrix_df.shape[0],
            'matrix_column_count': matrix_df.shape[1],
            'source_feature_count': feature_dictionary_df['source_field_name'].nunique(),
        }
    ]
)

feature_type_distribution_df = (
    feature_dictionary_df[['source_field_name', 'feature_type']]
    .drop_duplicates()
    .groupby('feature_type', as_index=False)
    .size()
    .rename(columns={'size': 'source_field_count'})
    .sort_values(['source_field_count', 'feature_type'], ascending=[False, True])
    .reset_index(drop=True)
)

encoding_plan_df = (
    feature_dictionary_df[['source_field_name', 'encoding_rule']]
    .drop_duplicates()
    .groupby('encoding_rule', as_index=False)
    .size()
    .rename(columns={'size': 'source_field_count'})
    .sort_values(['source_field_count', 'encoding_rule'], ascending=[False, True])
    .reset_index(drop=True)
)

missingness_plan_df = missingness_actions_df.copy()
manual_review_fields_df = (
    feature_dictionary_df.loc[
        feature_dictionary_df['manual_review_needed'] == 'true',
        ['source_field_name', 'feature_type', 'encoding_rule', 'missingness_action'],
    ]
    .drop_duplicates()
    .sort_values('source_field_name')
    .reset_index(drop=True)
)
matrix_preview_df = matrix_df.iloc[:10, : min(40, matrix_df.shape[1])].copy()
summary_review_df = summary_df.copy()

feature_dictionary_review_path = results_root / '91_baseline_model_input_v1_feature_dictionary.tsv'
encoding_spec_review_path = results_root / '92_baseline_model_input_v1_encoding_spec.tsv'
missingness_actions_review_path = results_root / '93_baseline_model_input_v1_missingness_actions.tsv'
matrix_preview_path = results_root / '94_baseline_model_input_v1_matrix_preview.tsv'
manual_review_path = results_root / '95_baseline_model_input_v1_manual_review_fields.tsv'
summary_review_path = results_root / '96_baseline_model_input_v1_summary.tsv'

feature_dictionary_df.to_csv(feature_dictionary_review_path, sep='\t', index=False)
encoding_spec_df.to_csv(encoding_spec_review_path, sep='\t', index=False)
missingness_plan_df.to_csv(missingness_actions_review_path, sep='\t', index=False)
matrix_preview_df.to_csv(matrix_preview_path, sep='\t', index=False)
manual_review_fields_df.to_csv(manual_review_path, sep='\t', index=False)
summary_review_df.to_csv(summary_review_path, sep='\t', index=False)


In [ ]:
print(f"Baseline model-input run ID: {latest_pointer['baseline_model_input_v1_run_id']}")
print(f"Baseline feature-set run ID: {latest_pointer['baseline_feature_set_v1_run_id']}")
print(f"Run log: {run_log_path}")
print(f"Saved: {feature_dictionary_review_path}")
print(f"Saved: {encoding_spec_review_path}")
print(f"Saved: {missingness_actions_review_path}")
print(f"Saved: {matrix_preview_path}")
print(f"Saved: {manual_review_path}")
print(f"Saved: {summary_review_path}")

display(pd.DataFrame([latest_pointer]))
display(pd.DataFrame([run_log.get('validation', {})]))
display(matrix_shape_df)
display(feature_type_distribution_df)
display(encoding_plan_df)
display(missingness_plan_df)
display(manual_review_fields_df)
display(summary_review_df)
